In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Drive'daki zip dosyalarını Colab'in hızlı yerel diskine çıkarıyoruz
%unzip -q /content/drive/MyDrive/cleaned_dataset.zip -d /content/
%unzip -q /content/drive/MyDrive/dataset_thermal_camera.zip -d /content/

In [ ]:
%pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 80.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 37.3 MB/s eta 0:00:00


In [ ]:
yaml_content = """
path: /content

train: cleaned_dataset/verified/images
val: dataset_thermal_camera/val/images
test: dataset_thermal_camera/test/images

names:
  0: human
"""

with open('/content/thermal_gold.yaml', 'w') as f:
    f.write(yaml_content.strip())

In [ ]:
import shutil
from ultralytics import YOLO

# 1. Modeli yükle
model = YOLO('yolov8n.pt')

# 2. Gold Model Eğitimini Başlat
results = model.train(
    data='/content/thermal_gold.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    workers=2,
    patience=20,
    name='thermal_yolov8_gold_best',
    device=0,
    plots=True  # Bütün PR, F1, Loss ve Confusion Matrix grafiklerini üretir
)

# 3. Sonuç Grafiklerini ve En İyi Ağırlığı Google Drive'a Kaydet
drive_save_path = '/content/drive/MyDrive/thermal_gold_results'

# Klasörü Drive üzerinde oluştur ve çıktıları kopyala
shutil.copytree('/content/runs/detect/thermal_yolov8_gold_best', drive_save_path, dirs_exist_ok=True)

print(f"\nEğitim ve tüm grafikler başarıyla Drive'a kaydedildi: {drive_save_path}")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/thermal_gold.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, fo

In [ ]:
import os
from ultralytics import YOLO

# Eğitilen en iyi modeli yükle
best_model = YOLO('/content/runs/detect/thermal_yolov8_gold_best/weights/best.pt')

# A) Test Kümesi Üzerinde Metrik Değerlendirmesi (Benchmark / Evaluation)
print("=== TEST KÜMESİ METRİK DEĞERLENDİRMESİ ===")
metrics = best_model.val(
    data='/content/thermal_gold.yaml',
    split='test',  # thermal_gold.yaml içindeki 'test' yolunu kullanır
    imgsz=640,
    batch=16,
    device=0,
    name='thermal_gold_test_eval'
)

print(f"Test mAP50: {metrics.box.map50:.4f}")
print(f"Test mAP50-95: {metrics.box.map:.4f}")
print(f"Test Precision: {metrics.box.mp:.4f}")
print(f"Test Recall: {metrics.box.mr:.4f}")

# B) Test Görselleri Üzerinde Tahmin Yapma ve Kutulu Görselleri Kaydetme (Inference)
print("\n=== TEST GÖRSELLERİ ÜZERİNDE TAHMİN YAPILIYOR ===")
results = best_model.predict(
    source='/content/dataset_thermal_camera/test/images', # Test görsellerinizin dizini
    conf=0.25,                                            # Güven eşiği
    save=True,                                            # Tahminli görselleri kaydeder
    name='thermal_gold_test_predictions',
    device=0
)

# C) Test Sonuçlarını da Drive'a Yedekle
shutil.copytree('/content/runs/detect/thermal_gold_test_eval', '/content/drive/MyDrive/thermal_gold_results/test_eval', dirs_exist_ok=True)
shutil.copytree('/content/runs/detect/thermal_gold_test_predictions', '/content/drive/MyDrive/thermal_gold_results/test_predictions', dirs_exist_ok=True)

print("\nTest sonuçları ve örnek tahmin görselleri Drive'a aktarıldı!")

=== TEST KÜMESİ METRİK DEĞERLENDİRMESİ ===
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 19.8±16.0 MB/s, size: 75.5 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning /content/dataset_thermal_camera/test/labels... 3522 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3522/3522 289.6it/s 12.2s
val: New cache created: /content/dataset_thermal_camera/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 221/221 5.9it/s 37.4s
                   all       3522       8543      0.946      0.887      0.951       0.62
Speed: 1.2ms preprocess, 3.1ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /content/runs/detect